# GPU stage 1 -- AMP check, new backbones, Waterbirds with model selection

Three things, ~1.1 h on an L4. Each persists to Drive on its own, so a lost session costs only the
stage in flight.

1. **AMP A/B on the GroupDRO arm.** The earlier A/B on the ERM arm found fp32 higher by 0.028-0.036,
   systematically. What that does *not* tell us is whether the shift is common-mode across
   representations -- if it is, every difference we report is unaffected. Four minutes settles it.
2. **Extract the two new frozen backbones** (R1.3). No training: one cached forward pass each.
3. **Re-run Waterbirds with model selection.** Every arm re-trains: `select_by` is part of
   `cache_key`, because which checkpoint is kept is part of the representation. The pre-selection
   caches keep their old names and stay on Drive as a with/without ablation.

## 0. Parameters -- **EDIT THESE**

In [ ]:
REPO_URL      = "https://github.com/octadion/vgscp"
REPO_BRANCH   = "main"
DRIVE_CACHE   = "/content/drive/MyDrive/vgscp_cache"
WATERBIRDS_URL= "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
CELEBA_SOURCE = "kaggle"       # needs kaggle.json in /content
CELEBA_DRIVE  = ""

# --- model selection: the protocol this study was missing --------------------------
# Sagawa et al. (2020), Kirichenko et al. (2023) and Liu et al. (2021) all select the checkpoint
# on worst-group VALIDATION accuracy, because these objectives overfit the minority group. Taking
# the last epoch instead let GroupDRO on CelebA drive train worst-group accuracy to 0.978 while
# evaluation stayed at ERM's level. Part of cache_key: which checkpoint is kept IS part of the
# representation, so turning this off gives different files rather than silently reusing these.
SELECT_BY          = "val_worst_group"
VAL_FRAC           = 0.1
VAL_MIN_PER_GROUP  = 15        # floor: 10% of Waterbirds' 56-example group is 6, SE ~0.20

FT_OBJECTIVES = ("erm", "groupdro", "reweight")
FT_HP = {
    "erm":      dict(optimizer="adam", lr=1e-3, weight_decay=0.0),
    "reweight": dict(optimizer="adam", lr=1e-3, weight_decay=0.0),
    "groupdro": dict(optimizer="sgd",  lr=1e-3, weight_decay=1e-2, groupdro_eta=0.05),
}
# Per (dataset, objective) and never shared: CelebA's train split is 34x Waterbirds', so one epoch
# is ~5.7 min there against seconds here.
FT_EPOCHS = {
    ("waterbirds", "erm"): 10, ("waterbirds", "reweight"): 10, ("waterbirds", "groupdro"): 20,
    ("celeba",     "erm"):  6, ("celeba",     "reweight"):  6, ("celeba",     "groupdro"):  6,
}
BATCH_SIZE    = 128
NUM_WORKERS   = None           # auto: cpu_count-1, capped at 12. Measured: no gain past ~8.
EXTRACT_BS    = 128            # fixed for the whole study; see finetune.py
EXTRACT_INFLIGHT = 3_500_000_000
AMP           = True           # fp32 weights/optimizer/loss, fp16 conv+matmul. In cache_key.
CACHE_DTYPE   = "float16"
N_SPLITS      = 10
HEADS         = ("erm", "dfr", "groupdro_ll")
SCORES        = ("APS", "RAPS", "THR")

# --- this notebook -----------------------------------------------------------------
RUN_DATASETS    = ("waterbirds",)  # this notebook trains Waterbirds only; the gate audits
                                   # exactly what will run, not both stages
EXPECT_TRAINING = True             # every arm retrains: adding model selection changed cache_key,
                                   # which is the point. The pre-selection caches stay on Drive.
FT_SEEDS   = (0, 1, 2, 3, 4)   # 5 seeds (R1.4); Waterbirds is ~2-4 min/run
NEW_FROZEN = ("dinov2_vitb14", "vit_b16_in1k")   # R1.3, extraction only

## 1. Drive + repo

In [ ]:
import os, sys, time, subprocess

def sh(cmd, check=True):
    """Show failures. A silent helper is how a Drive error becomes a wrong result three cells on."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout[-2000:])
    if r.returncode != 0:
        print(f"[shell FAILED rc={r.returncode}] {cmd}")
        if r.stderr.strip(): print(r.stderr[-2000:])
        if check: raise RuntimeError(f"command failed: {cmd}")
    return r.returncode == 0

def init_drive(mount="/content/drive", retries=3):
    """Mount Drive and PROVE it serves I/O -- the mount point can exist while every write fails."""
    from google.colab import drive
    for attempt in range(1, retries + 1):
        try:
            drive.mount(mount, force_remount=attempt > 1)
            os.makedirs(DRIVE_CACHE, exist_ok=True)
            probe, tok = os.path.join(DRIVE_CACHE, ".mount_probe"), str(time.time())
            with open(probe, "w") as fh: fh.write(tok)
            with open(probe) as fh: got = fh.read()
            os.remove(probe)
            if got != tok: raise IOError("probe read-back mismatch")
            free = os.statvfs(mount).f_bavail * os.statvfs(mount).f_frsize / 1e9
            print(f"Drive OK (attempt {attempt}) | ~{free:.1f} GB free")
            return
        except Exception as e:
            print(f"[drive] attempt {attempt}/{retries}: {e}"); time.sleep(5 * attempt)
    raise RuntimeError("Drive would not mount. Runtime > Disconnect and delete runtime, retry.")

init_drive()
REPO_DIR = "/content/vgscp"
sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

_cpus = os.cpu_count() or 4
if NUM_WORKERS is None:
    NUM_WORKERS = max(2, min(12, _cpus - 1))
    print(f"NUM_WORKERS auto -> {NUM_WORKERS} ({_cpus} vCPUs)")
elif NUM_WORKERS > _cpus:
    print(f"[warn] NUM_WORKERS {NUM_WORKERS} > {_cpus} vCPUs; clamping"); NUM_WORKERS = _cpus
print("repo:", os.getcwd())

## 2. Drive-backed caches

In [ ]:
os.makedirs("results", exist_ok=True)
for c in ("cache_clip", "cache_resnet", "cache_frozen", "cache_finetune", "study"):
    tgt = f"{DRIVE_CACHE}/{c}"; os.makedirs(tgt, exist_ok=True)
    sh(f"rm -rf results/{c}"); sh(f"ln -s {tgt} results/{c}")
    probe = f"results/{c}/.link_probe"                       # prove the link resolves onto Drive
    with open(probe, "w") as fh: fh.write("ok")
    assert os.path.exists(f"{tgt}/.link_probe"), f"results/{c} does not resolve to Drive"
    os.remove(probe)
CKPT_DIR = "/content/ft_ckpt"; os.makedirs(CKPT_DIR, exist_ok=True)   # local: Drive-write churn
print("caches symlinked to Drive and verified")
free = os.statvfs("/content/drive").f_bavail * os.statvfs("/content/drive").f_frsize / 1e9
print(f"Drive free: {free:.1f} GB")

## 3. Datasets

In [ ]:
from study_robust_train.colab_data import prepare_waterbirds, prepare_celeba
os.environ["WATERBIRDS_ROOT"] = prepare_waterbirds(DRIVE_CACHE, WATERBIRDS_URL)
CELEBA_ROOT = prepare_celeba(DRIVE_CACHE, source=CELEBA_SOURCE, celeba_drive=CELEBA_DRIVE)
CELEBA_OK = bool(CELEBA_ROOT) and os.path.isdir(CELEBA_ROOT)
if CELEBA_OK: os.environ["CELEBA_ROOT"] = CELEBA_ROOT
print("WATERBIRDS_ROOT =", os.environ["WATERBIRDS_ROOT"], "| CelebA OK =", CELEBA_OK)

## 4. Gate 1 -- analysis machinery

In [ ]:
rc = subprocess.run([sys.executable, "-m", "study_robust_train.validate_representation"],
                    capture_output=True, text=True)
print(rc.stdout[-2500:])
if rc.stderr.strip(): print(rc.stderr[-1200:])
assert rc.returncode == 0, "analysis validator FAILED"

## 5. Definitions (no training)

In [ ]:
from study_robust_train.representation import (build_repr_griddata,
                                                 run_representation_streaming,
                                                 write_representation_csv,
                                                 records_from_representation_csv,
                                                 write_representation_md)
from IPython.display import Markdown, display

REPR_CSV = "results/study/representation_records.csv"

def cfg_for(dataset, max_train=None):
    base = {"finetune": {"device": "cuda", "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS,
                         "amp": AMP, "max_train": max_train, "cache_dtype": CACHE_DTYPE,
                         "cache_dir": "results/cache_finetune", "ckpt_dir": CKPT_DIR,
                         "extract_batch_size": EXTRACT_BS,
                         "extract_inflight_bytes": EXTRACT_INFLIGHT,
                         "select_by": SELECT_BY, "val_frac": VAL_FRAC,
                         "val_min_per_group": VAL_MIN_PER_GROUP}}
    if dataset == "waterbirds":
        base["dataset"] = {"root": os.environ["WATERBIRDS_ROOT"], "image_size": 224,
                           "n_classes": 2, "download": False}
    else:
        base["dataset"] = {"root": os.environ["CELEBA_ROOT"], "n_classes": 2}
    return base

def keys_for(dataset, seeds):
    return [(dataset, o, s) for o in FT_OBJECTIVES for s in seeds]

def builder(dataset, max_train=None):
    cfg = cfg_for(dataset, max_train)
    def build(ds, obj, seed):
        t = time.time()
        gd = build_repr_griddata(ds, obj, cfg, ft_seed=seed,
                                 epochs=FT_EPOCHS[(ds, obj)], **FT_HP.get(obj, {}))
        print(f"[built] {ds}/{obj}/s{seed}  ({(time.time()-t)/60:.1f} min)", flush=True)
        return gd
    return build

def show_plan(dataset, seeds, max_train=None):
    print(f"[plan] {dataset}: "
          + ", ".join(f"{o}x{len(seeds)}@{FT_EPOCHS[(dataset,o)]}ep" for o in FT_OBJECTIVES)
          + f" | max_train={max_train} | select_by={SELECT_BY!r}")

print("defined -- no training yet")

## 6. Gate 2 -- pre-flight on the LIVE config\n\nAudits the values actually in scope, prints the predicted cache keys, and reports how many arms are already cached -- a miss means training, which on a CPU runtime is catastrophically slow with no warning.

In [ ]:
from study_robust_train.preflight_representation import audit
assert SELECT_BY, ("SELECT_BY is empty: that reverts to the last-epoch protocol this revision "
                   "exists to fix. Set it back to 'val_worst_group'.")
assert audit(globals()), "PRE-FLIGHT FAILED -- fix the config above before spending GPU time"

## 7. AMP A/B on the GroupDRO arm (~4 min)

`amp=True` is a cache hit if that arm was already built; only the fp32 twin costs anything.

**Read the per-head differences, not just the max.** If they are all close to the ERM arm's
(-0.031, -0.036, -0.028) the shift is common-mode and cancels in every comparison we report, so AMP
stays. If GroupDRO shifts by a *different* amount, the differences themselves move and we rebuild
with `AMP = False`.

In [ ]:
from study_robust_train.heads import head_probs
from study_robust_train.methods import fit_method
from study_robust_train import metrics
import numpy as np

cfg = cfg_for("waterbirds")
res = {}
for use_amp in (True, False):
    t = time.time()
    gd = build_repr_griddata("waterbirds", "groupdro", cfg, ft_seed=0,
                             epochs=FT_EPOCHS[("waterbirds", "groupdro")],
                             amp=use_amp, **FT_HP["groupdro"])
    Xev, yev, gev = gd.eval_domain
    res["amp" if use_amp else "fp32"] = {
        h: metrics.worst_group_accuracy(
               np.argmax(head_probs(fit_method(h, gd.train, gd.reweight, seed=0),
                                    Xev, gd.n_classes), axis=1), yev, gev)[1]
        for h in HEADS}
    print(f"  {'AMP ' if use_amp else 'fp32'} in {(time.time()-t)/60:.1f} min")
    del gd

ERM_ARM = {"erm": -0.031, "dfr": -0.036, "groupdro_ll": -0.028}   # measured earlier
print()
print(f"{'head':14s} {'AMP':>8s} {'fp32':>8s} {'diff':>8s} {'erm-arm':>9s} {'gap':>7s}")
gaps = []
for h in HEADS:
    a, f = res["amp"][h], res["fp32"][h]
    gap = (a - f) - ERM_ARM[h]; gaps.append(abs(gap))
    print(f"{h:14s} {a:8.3f} {f:8.3f} {a-f:+8.3f} {ERM_ARM[h]:+9.3f} {gap:+7.3f}")
print()
print(f"max |gap vs the ERM arm| = {max(gaps):.3f}  ->",
      "COMMON-MODE: differences are unaffected, keep AMP" if max(gaps) < 0.015
      else "NOT common-mode: set AMP=False and rebuild everything")

## 8. Extract the two new frozen backbones (~12 min)

Feature extraction only -- nothing is trained, and each is cached to Drive so the CPU grid can use
them without a GPU. Each model uses its own canonical preprocessing.

In [ ]:
from study_robust_train.datasets import _load_bundle, _features_for

fcfg = {"frozen": {"device": "cuda", "cache_dir": "results/cache_frozen",
                   "batch_size": EXTRACT_BS, "num_workers": NUM_WORKERS}}
targets = ["waterbirds"] + (["celeba"] if CELEBA_OK else [])
for ds in targets:
    cfg_ds = dict(fcfg); cfg_ds["dataset"] = cfg_for(ds)["dataset"]
    b = _load_bundle(ds, cfg_ds, 0)
    paths = {sp: b.meta["paths"][sp] for sp in ("train", "d_learn", "d_cal", "d_test")}
    y = {sp: np.asarray(b.y[sp]).astype(int) for sp in paths}
    for bb in NEW_FROZEN:
        t = time.time()
        feats = _features_for(bb, paths, y, cfg_ds, ds)
        dims = {sp: v.shape for sp, v in feats.items()}
        print(f"[{bb}/{ds}] {dims}  ({(time.time()-t)/60:.1f} min)", flush=True)
        del feats

## 9. Waterbirds with model selection (~50 min)

Watch two things per run:

* the **val slice** line -- it prints the per-group counts and the selection SE on the smallest
  group. Waterbirds' smallest training group has ~56 examples, so that SE is ~0.13 and the line is
  marked `[NOISY]`. Selection there is cliff-avoidance, not fine tuning; that is a limitation to
  state in the paper, not something to paper over.
* **`selected epoch k/N`** -- if selection consistently lands on the last epoch, it is doing
  nothing and the collapse we saw on CelebA was not an overfitting artefact.

In [ ]:
show_plan("waterbirds", FT_SEEDS)
wb_out = run_representation_streaming(keys_for("waterbirds", FT_SEEDS), builder("waterbirds"),
                                      heads=HEADS, scores=SCORES, n_splits=N_SPLITS)
print()
print("waterbirds records:", len(wb_out["records"]), "| failed:", wb_out["failed"])
write_representation_csv(wb_out["records"], REPR_CSV)
display(Markdown(write_representation_md(wb_out, "REPRESENTATION.md")))

## 10. STOP -- hand-off to GPU stage 2

`representation_records.csv` on Drive now holds the Waterbirds half. Stage 2 loads it and appends
CelebA, so the two sessions compose without either holding the other's features.

Before moving on, check the **manipulation check** at the top of the report: it must read PASS for
Waterbirds, i.e. a robust objective beat ERM at the *primary* head. If it reads WEAK, the
representation axis did not move and the CelebA budget should not be spent yet.

In [ ]:
sh(f"ls -la {DRIVE_CACHE}/study/", check=False)
print()
print("rows persisted:", len(records_from_representation_csv(REPR_CSV)))